In [1]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np

# Imports from PyTorch.
import torch
from torch import nn
from torchvision import datasets, transforms
from torch import nn, Tensor, device, no_grad, manual_seed
from torch import max as torch_max
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import sys
# get the path of the current file
file = os.path.abspath('')
path = os.path.join(file, '../../src')
sys.path.append(path)
print(path)


# Imports from aihwkit.
from aihwkit.utils.analog_info import analog_summary
from aihwkit.nn import AnalogConv2d, AnalogLinear, AnalogSequential
from aihwkit.optim import AnalogSGD
from aihwkit.simulator.configs.configs import (
    InferenceRPUConfig,
    # remember, the InferenceRPUConfig configuration parameter is used only for inference:
    # this means that the hardware non-idealities are considered only in the forward pass,
    # while the backward and update passes are ideal.
)
from aihwkit.simulator.configs import MappingParameter
from aihwkit.simulator.configs import (
    SingleRPUConfig,
    FloatingPointRPUConfig,
    ConstantStepDevice,
    FloatingPointDevice,
)

from aihwkit.simulator.parameters import (
    WeightQuantizerParameter,
)

from aihwkit.simulator.rpu_base import cuda



# Check device
USE_CUDA = 0
if cuda.is_compiled():
    USE_CUDA = 1

if USE_CUDA:
    torch.cuda.set_device(0)
DEVICE = torch.device("cuda:0" if USE_CUDA else "cpu")
print("Device: ", DEVICE)


cur_dir = os.getcwd()
print("Current directory: ", cur_dir)


# Path to store datasets
PATH_DATASET = os.path.join(cur_dir, "../data/svhn")
print("Path to store datasets: ", PATH_DATASET)



# Training parameters
SEED = 1
N_EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 0.1
N_CLASSES = 10

/home/ecabiati/cellar/aihwkit/sandbox/cuda/../../src
Device:  cuda:0
Current directory:  /home/ecabiati/cellar/aihwkit/sandbox/cuda
Path to store datasets:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn


In [2]:

WEIGHT_SCALING_OMEGA = 0.6  # Should not be larger than max weight.

# Select the device model to use in the training. In this case we are using one of the preset,
# but it can be changed to a number of preset to explore possible different analog devices
mapping = MappingParameter(weight_scaling_omega=WEIGHT_SCALING_OMEGA)

# RPU_CONFIG.runtime.offload_gradient = True
# RPU_CONFIG.runtime.offload_input = True


def load_images():
    """Load images for train from torchvision datasets."""
    mean = Tensor([0.4377, 0.4438, 0.4728])
    std = Tensor([0.1980, 0.2010, 0.1970])

    print(f"Normalization data: ({mean},{std})")

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
    train_set = datasets.SVHN(PATH_DATASET, download=True, split="train", transform=transform)
    val_set = datasets.SVHN(PATH_DATASET, download=True, split="test", transform=transform)
    train_data = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    validation_data = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    return train_data, validation_data


def create_analog_network(RPU_CONFIG):
    """Create a Vgg8 inspired analog model.

    Returns:
       nn.Module: VGG8 model
    """
    channel_base = 48
    channel = [channel_base, 2 * channel_base, 3 * channel_base]
    fc_size = 8 * channel_base
    model = AnalogSequential(
        AnalogConv2d(in_channels=3, out_channels=channel[0], kernel_size=3, stride=1, padding=1, rpu_config=RPU_CONFIG),
        nn.ReLU(),
        AnalogConv2d(
            in_channels=channel[0],
            out_channels=channel[0],
            kernel_size=3,
            stride=1,
            padding=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.BatchNorm2d(channel[0]),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1),
        AnalogConv2d(
            in_channels=channel[0],
            out_channels=channel[1],
            kernel_size=3,
            stride=1,
            padding=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.ReLU(),
        AnalogConv2d(
            in_channels=channel[1],
            out_channels=channel[1],
            kernel_size=3,
            stride=1,
            padding=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.BatchNorm2d(channel[1]),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1),
        AnalogConv2d(
            in_channels=channel[1],
            out_channels=channel[2],
            kernel_size=3,
            stride=1,
            padding=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.ReLU(),
        AnalogConv2d(
            in_channels=channel[2],
            out_channels=channel[2],
            kernel_size=3,
            stride=1,
            padding=1,
            rpu_config=RPU_CONFIG,
        ),
        nn.BatchNorm2d(channel[2]),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1),
        nn.Flatten(),
        AnalogLinear(in_features=16 * channel[2], out_features=fc_size, rpu_config=RPU_CONFIG),
        nn.ReLU(),
        AnalogLinear(in_features=fc_size, out_features=N_CLASSES, rpu_config=RPU_CONFIG),
        nn.LogSoftmax(dim=1),
    )
    return model

def create_sgd_optimizer(model, learning_rate):
    """Create the analog-aware optimizer.

    Args:
        model (nn.Module): model to be trained
        learning_rate (float): global parameter to define learning rate
    Returns:
        Optimizer: optimizer
    """
    optimizer = AnalogSGD(model.parameters(), lr=learning_rate)
    optimizer.regroup_param_groups(model)

    return optimizer


def train_step(train_data, model, criterion, optimizer):
    """Train network.

    Args:
        train_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer

    Returns:
        nn.Module, Optimizer, float: model, optimizer, and epoch loss
    """
    total_loss = 0

    model.train()

    for images, labels in train_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()

        # Add training Tensor to the model (input).
        output = model(images)
        loss = criterion(output, labels)

        # Run training (backward propagation).
        loss.backward()

        # Optimize weights.
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    epoch_loss = total_loss / len(train_data.dataset)

    return model, optimizer, epoch_loss


def test_evaluation(validation_data, model, criterion):
    """Test trained network

    Args:
        validation_data (DataLoader): Validation set to perform the evaluation
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss

    Returns:
        nn.Module, float, float, float: model, test epoch loss, test error, and test accuracy
    """
    total_loss = 0
    predicted_ok = 0
    total_images = 0

    model.eval()

    for images, labels in validation_data:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        pred = model(images)
        loss = criterion(pred, labels)
        total_loss += loss.item() * images.size(0)

        _, predicted = torch_max(pred.data, 1)
        total_images += labels.size(0)
        predicted_ok += (predicted == labels).sum().item()
        accuracy = predicted_ok / total_images * 100
        error = (1 - predicted_ok / total_images) * 100

    epoch_loss = total_loss / len(validation_data.dataset)

    return model, epoch_loss, error, accuracy


def training_loop(model, criterion, optimizer, train_data, validation_data, epochs, RESULTS,  print_every=1):
    """Training loop.

    Args:
        model (nn.Module): Trained model to be evaluated
        criterion (nn.CrossEntropyLoss): criterion to compute loss
        optimizer (Optimizer): analog model optimizer
        train_data (DataLoader): Validation set to perform the evaluation
        validation_data (DataLoader): Validation set to perform the evaluation
        epochs (int): global parameter to define epochs number
        print_every (int): defines how many times to print training progress

    Returns:
        nn.Module, Optimizer, Tuple: model, optimizer, and a tuple of
            lists of train losses, validation losses, and test error

    """
    train_losses = []
    valid_losses = []
    test_error = []

    # Train model
    for epoch in range(0, epochs):
        # Train_step
        model, optimizer, train_loss = train_step(train_data, model, criterion, optimizer)
        train_losses.append(train_loss)

        if epoch % print_every == (print_every - 1):
            # Validate_step
            with no_grad():
                model, valid_loss, error, accuracy = test_evaluation(
                    validation_data, model, criterion
                )
                valid_losses.append(valid_loss)
                test_error.append(error)

            print(
                f"{datetime.now().time().replace(microsecond=0)} --- "
                f"Epoch: {epoch}\t"
                f"Train loss: {train_loss:.4f}\t"
                f"Valid loss: {valid_loss:.4f}\t"
                f"Test error: {error:.2f}%\t"
                f"Test accuracy: {accuracy:.2f}%\t"
            )

    # Save results and plot figures
    np.savetxt(os.path.join(RESULTS, "Test_error.csv"), test_error, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Train_Losses.csv"), train_losses, delimiter=",")
    np.savetxt(os.path.join(RESULTS, "Valid_Losses.csv"), valid_losses, delimiter=",")
    plot_results(train_losses, valid_losses, test_error, RESULTS)

    return model, optimizer, (train_losses, valid_losses, test_error)


def plot_results(train_losses, valid_losses, test_error, RESULTS):
    """Plot results.

    Args:
        train_losses (List): training losses as calculated in the training_loop
        valid_losses (List): validation losses as calculated in the training_loop
        test_error (List): test error as calculated in the training_loop
    """
    fig = plt.plot(train_losses, "r-s", valid_losses, "b-o")
    plt.title("aihwkit VGG8")
    plt.legend(fig[:2], ["Training Losses", "Validation Losses"])
    plt.xlabel("Epoch number")
    plt.ylabel("Loss [A.U.]")
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_losses.png"))
    plt.close()

    fig = plt.plot(test_error, "r-s")
    plt.title("aihwkit VGG8")
    plt.legend(fig[:1], ["Test Error"])
    plt.xlabel("Epoch number")
    plt.ylabel("Test Error [%]")
    plt.yscale("log")
    plt.ylim((5e-1, 1e2))
    plt.grid(which="both", linestyle="--")
    plt.savefig(os.path.join(RESULTS, "test_error.png"))
    plt.close()


In [3]:
def main_unquantized():
    """Train a PyTorch CNN analog model with the MNIST dataset."""
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    # Path to store results
    RESULTS = os.path.join(cur_dir, "vgg8_results")
    print("Path to store results: ", RESULTS)
    os.makedirs(RESULTS, exist_ok=True)
    manual_seed(SEED)

    # Load datasets.
    train_data, validation_data = load_images()

    # Select RPU_CONFIG
    # RPU_CONFIG = InferenceRPUConfig()
    RPU_CONFIG = FloatingPointRPUConfig()

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG)
    if USE_CUDA:
        model.cuda()
    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started Vgg8 Example")

    optimizer = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, train_data, validation_data, N_EPOCHS, RESULTS
    )
    
    # Save the model to .th file
    torch.save(model.state_dict(), os.path.join(RESULTS, "vgg8.th"))

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed Vgg8 Example")

    

def main_quantized():
    """Train a PyTorch CNN analog model with the MNIST dataset."""
    # Make sure the directory where to save the results exist.
    # Results include: Loss vs Epoch graph, Accuracy vs Epoch graph and vector data.
    # Path to store results
    RESULTS = os.path.join(cur_dir, "vgg8_quantized_results")
    print("Path to store results: ", RESULTS)

    os.makedirs(RESULTS, exist_ok=True)
    manual_seed(SEED)

    # Load datasets.
    train_data, validation_data = load_images()

    # Select RPU_CONFIG
    # RPU_CONFIG = InferenceRPUConfig()
    RPU_CONFIG = FloatingPointRPUConfig()

    RPU_CONFIG.quantization = WeightQuantizerParameter(
        resolution = 0.3,
        #amax_channelwise= True,
        eps= 0.25,
        levels = 3,
        method = "percentile",
        use_forward=True,
    )

    # Prepare the model.
    model = create_analog_network(RPU_CONFIG)
    if USE_CUDA:
        model.cuda()
    print(model)

    print(f"\n{datetime.now().time().replace(microsecond=0)} --- " f"Started Vgg8 Example")

    optimizer = create_sgd_optimizer(model, LEARNING_RATE)

    criterion = nn.CrossEntropyLoss()

    model, optimizer, _ = training_loop(
        model, criterion, optimizer, train_data, validation_data, N_EPOCHS, RESULTS
    )

    # Save the model to .th file
    torch.save(model.state_dict(), os.path.join(RESULTS, "vgg8_quant.th"))

    print(f"{datetime.now().time().replace(microsecond=0)} --- " f"Completed Vgg8 Example")

In [ ]:


# Plot a single instance of the dataset
train_data, test_data = load_images()
sample, _ = next(iter(train_data))
sample = sample[0]
sample = sample.permute(1, 2, 0).numpy()
print(" Sample dimensions: ", sample.shape)
# Standardize the sample values in the range [0, 1]
#sample = (sample - sample.min()) / (sample.max() - sample.min())
plt.imshow(sample)
plt.show()



        


Normalization data: (tensor([0.4377, 0.4438, 0.4728]),tensor([0.1980, 0.2010, 0.1970]))


In [5]:
RPU_CONFIG = FloatingPointRPUConfig()

# Prepare the model.
model = create_analog_network(RPU_CONFIG)
analog_summary(model, input_size=(1, 3, 32, 32))


Model Name: AnalogSequential
Per-layer Information
Layer Information                                                     | Tile Information              
Layer Name          Is Analog           In Shape            Out Shape           Kernel Shape        # of Tiles          Reuse Factor        Log. tile shape     Phys. tile shape    utilization (%)     
AnalogConv2d        analog              [1, 3, 32, 32]      [1, 48, 32, 32]     (3, 3)              1                   1024                -                   -                   -                   
                                                                                                                                            (48, 27)            N/A                 100.00              
ReLU                digital             [1, 48, 32, 32]     [1, 48, 32, 32]     -                   0                   0                   -                   -                   -                   
AnalogConv2d        analog              [1

In [6]:
main_unquantized()

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/vgg8_results
Normalization data: (tensor([0.4377, 0.4438, 0.4728]),tensor([0.1980, 0.2010, 0.1970]))
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/train_32x32.mat
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/test_32x32.mat
AnalogSequential(
  (0): AnalogConv2d(
    3, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(48,27))
  )
  (1): ReLU()
  (2): AnalogConv2d(
    48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(48,432))
  )
  (3): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): AnalogConv2d(
    48, 96, kernel_size

In [7]:
main_quantized()

Path to store results:  /home/ecabiati/cellar/aihwkit/sandbox/cuda/vgg8_quantized_results
Normalization data: (tensor([0.4377, 0.4438, 0.4728]),tensor([0.1980, 0.2010, 0.1970]))
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/train_32x32.mat
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/test_32x32.mat
AnalogSequential(
  (0): AnalogConv2d(
    3, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(48,27))
  )
  (1): ReLU()
  (2): AnalogConv2d(
    48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), FloatingPointRPUConfig
    (analog_module): FloatingPointTile(RPUCudaSimple<float>(48,432))
  )
  (3): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): AnalogConv2d(
    48, 96, k

In [4]:
from aihwkit.nn.conversion import convert_to_analog

# Load datasets.
_, validation_data = load_images()

criterion = nn.CrossEntropyLoss()

# For the instantiation, the RPU config object should be OK to be the same for
# the two models
# rpu_config = InferenceRPUConfig()
rpu_config = FloatingPointRPUConfig()

# Standard model
RESULTS_NON_QUANTIZED = cur_dir + "/vgg8_results"
state_dict = torch.load(os.path.join(RESULTS_NON_QUANTIZED, "vgg8.th"), map_location=DEVICE)
non_quantized_model = create_analog_network(rpu_config).to(DEVICE)
non_quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)

# QA model
RESULTS_QUANTIZED = cur_dir + "/vgg8_quantized_results"
state_dict = torch.load(os.path.join(RESULTS_QUANTIZED, "vgg8_quant.th"), map_location=DEVICE)
quantized_model = create_analog_network(rpu_config).to(DEVICE)
quantized_model.load_state_dict(state_dict, strict=True, load_rpu_config=False)

# apply quantization on the QA model, the same for which it was trained
rpu_config.quantization = WeightQuantizerParameter(
        resolution = 0.3,
        #amax_channelwise= True,
        eps= 0.25,
        levels = 3,
        method = "percentile"
    )

#non_quantized_model = convert_to_analog(non_quantized_model, rpu_config)
quantized_model = convert_to_analog(quantized_model, rpu_config)


# Evaluate the models
non_quantized_model.eval()
quantized_model.eval()

_, _, _, accuracy_non_quantized = test_evaluation(validation_data, non_quantized_model, criterion)
_, _, _, accuracy_quantized = test_evaluation(validation_data, quantized_model, criterion)

print(f"Accuracy of the non-quantized model: {accuracy_non_quantized:.2f}%")
print(f"Accuracy of the quantized model: {accuracy_quantized:.2f}%")


# Plot the weights of the first layer
src = os.getcwd()
src = os.path.abspath(os.path.join(src, '../src'))
sys.path.append(src)

import plotting as pl

pl.generate_moving_hist(non_quantized_model, title = "Non-quantized model", file_name = RESULTS_NON_QUANTIZED + "/weights_hist_non_quantized.gif", range = (-1, 1), top = None, split_by_rows=False)
pl.generate_moving_hist(quantized_model, title = "Quantized model", file_name = RESULTS_QUANTIZED + "/weights_hist_quantized.gif", range = (-1, 1), top = None, split_by_rows=False)








Normalization data: (tensor([0.4377, 0.4438, 0.4728]),tensor([0.1980, 0.2010, 0.1970]))
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/train_32x32.mat
Using downloaded and verified file: /home/ecabiati/cellar/aihwkit/sandbox/cuda/../data/svhn/test_32x32.mat
Accuracy of the non-quantized model: 94.89%
Accuracy of the quantized model: 92.66%


<Figure size 1500x600 with 0 Axes>

<Figure size 1500x600 with 0 Axes>